# AI usage, quota and capacity

This notebook analyzes locally supplied usage and quota aggregates. It does not fetch evidence or make provider requests. Input files belong in the ignored `evidence/` directory; source watermarks are displayed after loading.

We distinguish recorded weekly throughput, quota-percent burn, and conditional token capacity per 100 quota points. Random resets and redeemed awards can increase observed throughput without increasing the regular subscription allowance.

Quota figures and tables focus on **weekly budgets** following the Claude Max 20× → Ultra tier change. [Verified Ultra accounts have no five-hour limit](https://support.claude.com/en/articles/17728952-what-is-the-ultra-plan). Historical five-hour polls can remain in calibration evidence.

Change the parameters below and run all cells to recompute. Parsing, range selection, regression and resampling live in the adjacent Python modules.


In [ ]:
from pathlib import Path
from datetime import timedelta
import json
import sys

import numpy
import pandas
import matplotlib.pyplot
import matplotlib.dates
from IPython.display import display

PROJECT_DIR = Path.cwd()
EVIDENCE = PROJECT_DIR / "evidence"
sys.path.insert(0, str(PROJECT_DIR.parents[1]))

from x.ai_sub_value.analyze import Poll, ObservedRate, measure_hour, parse_time, read_rows
from x.ai_sub_value.estimate_capacity import build_estimates, load_bins, read_money
from x.ai_sub_value.report_data import quota_weekly
from x.ai_sub_value.plan_history import codex_plan_at, read_codex_plan_history

%matplotlib inline
matplotlib.pyplot.rcParams.update({"figure.dpi": 115, "axes.spines.top": False, "axes.spines.right": False})
pandas.set_option("display.max_rows", 25)
pandas.set_option("display.max_columns", 20)

## Parameters

`CHUNK_MINUTES` is an elapsed-time target **within each monotonic run**, anchored at that run's first real poll. Every short prefix/tail is retained, and no calibration interval crosses a detected discontinuity. Whole-run, 30-, 60- and 120-minute point fits are compared separately below.

Bootstrap blocks and regression aggregation are different choices. Day bootstrap methods preserve groups of calibration observations; they do not estimate consumption across resets.

In [ ]:
CHUNK_MINUTES = 60
BOOTSTRAP_SAMPLES = 500
SENSITIVITY_SAMPLES_PER_SEED = 2000
RECENT_WEEKS = 12
QUOTA_HISTORY_DAYS = 14

In [ ]:
quota_rows = read_rows(EVIDENCE / "quota-polls.jsonl.gz")
codex_plan_periods = read_codex_plan_history(EVIDENCE / "codex-plan-history.json")
scoped_rows = read_rows(EVIDENCE / "claude-scoped-quota.jsonl.gz")
accounting_rows = read_rows(EVIDENCE / "raw-accounting.jsonl.gz")
bins, input_audit = load_bins([
    EVIDENCE / "native-usage-5m.json.gz",
    EVIDENCE / "cloud-usage-5m.json.gz",
    EVIDENCE / "langfuse-usage-five-minute.json.gz",
])
weekly = pandas.read_csv(EVIDENCE / "weekly-model-usage.csv", parse_dates=["week"])
manifest = json.loads((EVIDENCE / "usage-accounting-manifest.json").read_text())
display(pandas.Series(manifest["source_watermarks"], name="source watermark (UTC)").to_frame())
display(pandas.Series({key: value for key, value in input_audit.items() if key != "source_files"}, name="input accounting").to_frame())

## Weekly recorded tokens by model and source

Sources stay separate because cross-system request overlap is unresolved. These are recorded totals, including whatever resets or paid usage is represented in each source. The latest week is partial. Unknown token totals remain missing; plotted sums contain known observations only.

Each provider uses one shared Y scale across its source panels. OpenAI has no Claude Code Web path.

In [ ]:
first_week = weekly["week"].max() - pandas.Timedelta(weeks=RECENT_WEEKS - 1)
recent = weekly[weekly["week"] >= first_week]
provider_sources = {
    "openai": ["native", "langfuse"],
    "anthropic": ["native", "claude_code_web", "langfuse"],
}
for provider, sources in provider_sources.items():
    fig, axes = matplotlib.pyplot.subplots(1, len(sources), figsize=(6 * len(sources), 5), sharey=True, sharex=True, squeeze=False)
    provider_max = recent[recent.provider == provider].groupby(["source_group", "week", "model"])["total_tokens"].sum(min_count=1).max() / 1e9
    for ax, source in zip(axes[0], sources):
        subset = recent[(recent.provider == provider) & (recent.source_group == source)]
        table = subset.pivot_table(index="week", columns="model", values="total_tokens", aggfunc=lambda x: x.sum(min_count=1))
        table = table.reindex(pandas.date_range(first_week, weekly.week.max(), freq="W-MON"))
        top = table.sum().nlargest(6).index
        for model in top:
            ax.plot(table.index.to_pydatetime(), table[model].to_numpy() / 1e9, marker=".", linewidth=1.5, label=model)
        ax.axvspan(weekly.week.max(), weekly.week.max() + pandas.Timedelta(days=7), alpha=.07, color="grey")
        locator = matplotlib.dates.AutoDateLocator(minticks=4, maxticks=6)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(matplotlib.dates.ConciseDateFormatter(locator))
        ax.set_title(source)
        ax.set_ylim(0, provider_max * 1.05)
        ax.grid(alpha=.2)
        ax.legend(fontsize=8)
    axes[0][0].set_ylabel("Billions of recorded tokens / UTC week")
    fig.suptitle(f"{provider} · same Y scale across sources · latest week is partial")
    fig.autofmt_xdate()
    fig.tight_layout()
    matplotlib.pyplot.show()


## Token categories in the latest captured week

Cache-read input dominates many recorded totals. Reasoning tokens are a subset of output and are not added again. The plot uses native rows with every displayed category known; excluded incomplete rows are counted below.

In [ ]:
category_columns = ["uncached_input_tokens", "cache_read_input_tokens", "cache_write_input_tokens", "output_tokens"]
latest_native = weekly[(weekly.week == weekly.week.max()) & (weekly.source_group == "native")]
complete = latest_native.dropna(subset=category_columns)
print(f"Latest native week: {weekly.week.max().date()} (partial); rows omitted for unknown categories: {len(latest_native) - len(complete)}")
mix = complete.groupby(["provider", "model"])[category_columns].sum().sort_values(category_columns, ascending=False)
mix = mix.loc[mix.sum(axis=1).nlargest(10).index]
(mix / 1e9).plot.barh(stacked=True, figsize=(12, 6), width=.8)
matplotlib.pyplot.xlabel("Billions of recorded tokens; known categories")
matplotlib.pyplot.ylabel("")
matplotlib.pyplot.legend(loc="lower right", fontsize=8)
matplotlib.pyplot.tight_layout()
matplotlib.pyplot.show()

## Recompute calibration from monotonic poll ranges

Adjacent poll spans are retained while the quota gauge is nondecreasing, reset metadata is consistent, the gauge is below saturation and polling gaps are at most 15 minutes. Companion gauges and stable paid-usage counters are screened at the same poll-span grain. Only the uncertain span across a detected break is excluded.

The existing 15-minute money-counter bracket is preserved. A reset that consumption completely masks between polls remains unidentifiable. Frontend equality and predominantly subscription traffic on native/Web sources remain working assumptions.
Codex fits are split by exact plan and quota-regime periods in `evidence/codex-plan-history.json`. The provider reports only the broad `plan_type: pro`, so record tier or included-quota changes and their effective timestamps there; the estimator rejects uncovered polls and periods that span a reported category change.

In [ ]:
capacity = build_estimates(
    quota_rows,
    bins,
    read_money(accounting_rows),
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    sensitivity_samples=SENSITIVITY_SAMPLES_PER_SEED,
    chunk_minutes=CHUNK_MINUTES,
    scoped_quota_rows=scoped_rows,
    codex_plan_periods=codex_plan_periods,
)
capacity["input_audit"] = input_audit
primary = [(result, result["scenarios"][0]) for result in capacity["results"] if result["window_seconds"] == 604800]
retention = []
for result, fit in primary:
    retention.append({
        "provider": result["provider"], "subscription plan": result.get("subscription_plan"), "window": "weekly",
        "scope": result["name"] or "general", "monotonic runs": len(result["monotonic_runs"]),
        "fit intervals": fit["intervals"], "observed hours": fit["observed_hours"],
        "days": fit["days"], "held-out R²": fit["validation"].get("held_out_r_squared"),
    })
display(pandas.DataFrame(retention).round(3))

## Inspect quota readings and retained runs

Grey points are all observed readings; blue lines show retained monotonic ranges. Gaps between blue segments include excluded reset, saturation, polling, companion-gauge or money-counter spans. The two providers' percentages belong to different budgets.

In [ ]:
fig, axes = matplotlib.pyplot.subplots(2, 1, figsize=(15, 8), sharex=True)
for ax, provider in zip(axes, ["codex", "claude"]):
    rows = sorted([r for r in quota_rows if r["provider"] == provider and r["window_seconds"] == 604800 and not r["window_name"]], key=lambda r: r["observed_at"])
    observed = pandas.DataFrame({"at": pandas.to_datetime([r["observed_at"] for r in rows], utc=True), "used": [r["used_percent"] for r in rows]})
    cutoff = observed["at"].max() - pandas.Timedelta(days=QUOTA_HISTORY_DAYS)
    shown = observed[observed["at"] >= cutoff]
    ax.scatter(shown["at"], shown.used, s=5, color="grey", alpha=.35, label="All polls")
    provider_results = [r for r, _ in primary if r["provider"] == provider and r["window_seconds"] == 604800 and not r["name"]]
    for result in provider_results:
        label = result.get("subscription_plan") or "Retained monotonic ranges"
        labelled = False
        for run in result["monotonic_runs"]:
            start, end = pandas.Timestamp(run["start"]), pandas.Timestamp(run["end"])
            subset = observed[(observed["at"] >= max(start, cutoff)) & (observed["at"] <= end)]
            if len(subset):
                ax.plot(subset["at"], subset.used, linewidth=1.4, label=label if not labelled else None)
                labelled = True
    ax.set_title(f"{provider} · weekly quota")
    ax.set_ylabel("Used quota (%)")
    ax.set_ylim(-2, 102)
    ax.grid(alpha=.2)
    ax.legend(loc="upper left")
fig.autofmt_xdate()
fig.tight_layout()
matplotlib.pyplot.show()

run_rows = [{"provider": r["provider"], "subscription_plan": r.get("subscription_plan"), "window_seconds": r["window_seconds"], "scope": r["name"], **run} for r, _ in primary for run in r["monotonic_runs"]]
ranges = pandas.DataFrame(run_rows)
ranges["duration_hours"] = (pandas.to_datetime(ranges.end, utc=True) - pandas.to_datetime(ranges.start, utc=True)).dt.total_seconds() / 3600
display(ranges.sort_values("duration_hours", ascending=False).head(12))

## Does combining a whole run change model attribution?

These point-fit diagnostics use exactly the same accepted poll spans. Thirty-, sixty- and 120-minute pieces all retain their short ends. Whole-run fitting aggregates each complete monotonic range into one equation. It can change leverage and lose separation between models used within a run. No confidence interval or independent-day support is asserted for these diagnostic whole-run coefficients.

In [ ]:
general = [(r, f) for r, f in primary if not r["name"]]
fig, axes = matplotlib.pyplot.subplots(1, max(1, len(general)), figsize=(6 * max(1, len(general)), 5), squeeze=False)
for ax, (result, fit) in zip(axes[0], general):
    supported = [m["model"] for m in fit["models"] if m["status"] == "conditional_estimate"]
    aggregation = result["aggregation_sensitivity"]
    baseline = next(item for item in aggregation if item["chunk_minutes"] == 60)
    base = {m["model"]: m["points_per_million_recorded_tokens"] for m in baseline["models"]}
    for model in supported:
        values = [next(m["points_per_million_recorded_tokens"] for m in item["models"] if m["model"] == model) / base[model] for item in aggregation]
        ax.plot(range(len(aggregation)), values, marker="o", label=model)
    ax.axhline(1, color="grey", linestyle="--", linewidth=1)
    ax.set_xticks(range(4), ["30 min", "60 min", "120 min", "Whole run"])
    ax.set_title(f"{result['provider']} · weekly quota")
    plan = result.get("subscription_plan")
    if plan: ax.set_title(f"{result['provider']} / {plan}")
    ax.set_ylabel("Fitted quota cost / 60-minute fit")
    ax.set_yscale("log")
    ax.grid(alpha=.2)
    if supported:
        ax.legend(fontsize=7)
fig.tight_layout()
matplotlib.pyplot.show()

aggregation_coverage = pandas.DataFrame([{ "provider": r["provider"], "subscription_plan": r.get("subscription_plan"), "window_seconds": r["window_seconds"], "scope": r["name"], **{k: v for k, v in item.items() if k != "models"}} for r, _ in primary for item in r["aggregation_sensitivity"]])
display(aggregation_coverage)

## Conditional tokens per weekly quota budget

Only models passing the primary model-separation, active-day, bootstrap and chronological prediction checks receive published capacity values. A total-token coefficient averages the observed cache/output/effort/speed mix; it is not an entitlement or a universal token price. The following intervals remain conditional on accounting, account scope and the selected statistical method.

In [ ]:
capacity_rows=[]
for result, fit in primary:
    for model in fit["models"]:
        interval=model["capacity_bootstrap_90_million_tokens"]
        capacity_rows.append({
            "provider":result["provider"], "subscription plan":result.get("subscription_plan"), "window": "weekly",
            "scope":result["name"] or "general", "model":model["model"], "qualification":model["status"],
            "point (B tokens)": model["million_recorded_tokens_per_100_points"] / 1000 if model["million_recorded_tokens_per_100_points"] is not None else numpy.nan,
            "90% lower (B)": interval[0] / 1000 if interval else numpy.nan,
            "90% upper (B)": interval[1] / 1000 if interval else numpy.nan,
            "active days":model["active_days"],
        })
capacity_table=pandas.DataFrame(capacity_rows)
display(capacity_table.round(3))

## Bootstrap block-length and endpoint sensitivity

Each provider has one figure containing its qualified models on a common **logarithmic cost axis**. Each figure uses the provider's weekly quota budget. Farther right means more quota consumed per recorded token. Bars span the 5th–95th percentiles: nominal, conditional 90% bootstrap intervals. Each color is a bootstrap method; black diamonds show the shared point fit. These are costs per million tokens, not costs relative to a baseline and not burn rates per hour.

The graphs retain models passing the primary qualification checks; the table above includes every model and the reason for any withheld estimate. Every method uses the same point fit and accepted intervals, with two seeds and the configured replication count per seed. Hour methods resample complete groups of intervals sharing a UTC start hour. Day methods retain all observations assigned to that day.

Moving blocks underweight endpoints. Circular methods balance calendar-day inclusion, retain empty calendar days and wrap the record's end to its beginning; that creates artificial adjacency at one seam. Longer blocks are not automatically conservative, especially with only a few weeks of data. A coefficient interval reaching zero is marked explicitly rather than plotted at zero on a log axis; it does not establish free or unlimited usage.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

method_names = {
    "independent_hours": "Independent hour groups",
    "six_hours": "Six-hour blocks",
    "one_day": "One-day blocks",
    "three_days": "Three-day blocks",
    "seven_days": "Seven-day blocks",
    "circular_three_days": "Circular three-day",
    "circular_seven_days": "Circular seven-day",
}
method_colors = dict(zip(method_names, matplotlib.pyplot.get_cmap("tab10").colors))
for provider in ("codex", "claude"):
    panels = [(result, fit) for result, fit in general if result["provider"] == provider]
    fig, axes = matplotlib.pyplot.subplots(1, len(panels), figsize=(8 * len(panels), 6.5), squeeze=False, sharex=True)
    positive_bounds = []
    for ax, (result, fit) in zip(axes[0], panels):
        supported = sorted(
            [model for model in fit["models"] if model["status"] == "conditional_estimate"],
            key=lambda model: model["points_per_million_recorded_tokens"],
        )
        methods = fit["block_sensitivity"]["methods"]
        offsets = numpy.linspace(-.28, .28, len(methods))
        for y, model in enumerate(supported):
            for offset, method in zip(offsets, methods):
                estimate = next(item for item in method["models"] if item["model"] == model["model"])
                lower, upper = estimate["coefficient_bootstrap_90"]
                color = method_colors[method["method"]]
                if upper > 0:
                    positive_bounds.append(upper)
                    if lower > 0:
                        positive_bounds.append(lower)
                        ax.plot([lower, upper], [y + offset, y + offset], color=color, linewidth=2, marker="|", markersize=5)
                    else:
                        ax.plot(upper, y + offset, color=color, marker="<", markersize=6)
                        ax.annotate("reaches zero", (upper, y + offset), xytext=(5, 0), textcoords="offset points", fontsize=7, va="center", color=color)
            ax.plot(model["points_per_million_recorded_tokens"], y, "D", color="black", markersize=4, zorder=5)
        ax.set_yticks(range(len(supported)), [model["model"] for model in supported])
        ax.set_ylim(-.6, max(len(supported) - .4, .6))
        ax.invert_yaxis()
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
        ax.set_title(result.get("subscription_plan") or "Weekly quota")
        ax.set_xlabel("Quota percentage points per million recorded tokens" + chr(10) + "(log scale; farther right = higher quota cost per token)")
        ax.grid(axis="x", which="both", alpha=.2)
    if positive_bounds:
        axes[0][0].set_xlim(min(positive_bounds) / 1.3, max(positive_bounds) * 1.3)
    handles = [Line2D([0], [0], color=method_colors[method], linewidth=2, label=label) for method, label in method_names.items()]
    handles.append(Line2D([0], [0], color="black", marker="D", linestyle="none", markersize=4, label="Shared point fit"))
    fig.suptitle(f"{provider} · model quota costs", y=.99)
    fig.text(.5, .945, "Conditional 90% percentile-bootstrap intervals (5th–95th percentiles)", ha="center", fontsize=10)
    fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(.5, .915), ncol=4, fontsize=8, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, .81))
    matplotlib.pyplot.show()


## Dependence of fitting errors

Contributions are summed by the UTC hour in which each calibration interval starts. Correlations use observed pairs at the stated elapsed-hour lag, without stitching missing hours. These are descriptive diagnostics, not significance tests; model-specific score correlations can differ from the aggregate residual pattern.

In [ ]:
fig,ax=matplotlib.pyplot.subplots(figsize=(11,4))
for result,fit in general:
    data=pandas.DataFrame(fit["block_sensitivity"]["residual_lag_correlations"])
    ax.plot(data.lag_hours,data.correlation,marker="o",label=f"{result['provider']} · weekly quota")
ax.axhline(0,color="grey",linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("Elapsed lag (hours)")
ax.set_ylabel("Correlation of start-hour residual sums")
ax.legend()
ax.grid(alpha=.2)
matplotlib.pyplot.tight_layout()
matplotlib.pyplot.show()

## Weekly-quota-percent burn per hour over time

Each point brackets a preceding hour using the recorded polls. Windows crossing detected resets, long polling gaps or saturation remain unavailable. Shading is a conditional gauge-rounding envelope, not a bootstrap confidence interval. A point of one provider's weekly gauge is not equivalent to a point of the other's.

In [ ]:
fig,axes=matplotlib.pyplot.subplots(2,1,figsize=(15,7),sharex=True)
for ax,(provider,window) in zip(axes,[("codex",604800),("claude",604800)]):
    selected=sorted([r for r in quota_rows if r["provider"]==provider and r["window_seconds"]==window and not r["window_name"]],key=lambda r:r["observed_at"])
    polls=[Poll(parse_time(r["observed_at"]),r["used_percent"],parse_time(r["reset_at"]) if r["reset_at"] else None) for r in selected]
    cutoff=polls[-1].at-timedelta(days=7)
    points=[]
    for i in range(24,len(polls),3):
        if polls[i].at<cutoff:
            continue
        if provider == "codex" and codex_plan_at(polls[i].at, codex_plan_periods) != codex_plan_at(polls[i].at - timedelta(hours=1), codex_plan_periods):
            continue
        rate=measure_hour(polls[max(0,i-24):i+1],cutoff=polls[i].at)
        if isinstance(rate,ObservedRate):
            points.append((polls[i].at,rate.points_per_hour,rate.lower,rate.upper))
        else:
            points.append((polls[i].at,numpy.nan,numpy.nan,numpy.nan))
    frame=pandas.DataFrame(points,columns=["at","rate","lower","upper"])
    ax.plot(frame["at"],frame.rate,linewidth=1)
    ax.fill_between(frame["at"],frame.lower,frame.upper,alpha=.15)
    ax.set_title(f"{provider} · weekly quota")
    ax.set_ylabel("Weekly quota percentage points / hour")
    ax.grid(alpha=.2)
fig.autofmt_xdate()
fig.tight_layout()
matplotlib.pyplot.show()

## Observed resets, restored headroom and accounting exclusions

Downward changes are recorded separately from consumption. They do not establish whether the cause was a scheduled reset, a redeemed award, a provider adjustment or another policy change. Consumed quota between polls can mask a reset completely.

In [ ]:
weekly_quota=pandas.DataFrame(quota_weekly(quota_rows))
display(weekly_quota[(weekly_quota.window_seconds==604800) & (weekly_quota.name=="")].tail(20))
display(pandas.DataFrame([{ "provider":r["provider"], "window_seconds":r["window_seconds"], "scope":r["name"], **r["rejected_poll_pairs"]} for r,_ in primary]).fillna(0))

## What these intervals do not cover

- Deleted or uncaptured transcripts, partial-record gaps, stale source discovery and unresolved cross-source duplicate identities.
- Unconfirmed billing/account attribution. Known API-key lanes are excluded from quota fitting; native/Web unknown lanes follow the user's predominantly subscription-traffic assumption.
- Unobserved resets, changes in allowance scale, and causal attribution of redeemed awards or extra paid usage.
- Independent cache, input, output, effort and speed prices. Current coefficients average the recorded mix.
- Future provider policies or workloads. Three days is a block-length scenario, not a validated optimum or a demonstrated coverage guarantee.
- Unexported web and direct API usage, whose volume must be measured or recorded as an explicit coverage assumption.

Weekly totals are captured counts, not a random population sample; they do not receive invented sampling confidence intervals. The union of captured records can still miss real usage.

In [ ]:
output = EVIDENCE / "monotonic-capacity-estimates.json"
output.write_text(json.dumps(capacity, indent=2, allow_nan=False) + "\n")
print(f"Saved recomputed monotonic-range results: {output.name}")